In [23]:
import pandas as pd
from pathlib import Path

In [24]:
BASE_DIR     = Path(r"C:\Users\hecto\OneDrive\Escritorio\Personal\iroFactory\35.LinNeo")
DATA_DIR     = BASE_DIR / "biodiversity_data"
TAXONOMY_CSV = DATA_DIR / "gbif_taxonomy.csv"
SPECIES_CSV  = DATA_DIR / "taxonomy_nodes_by_rank" / "species_nodes.csv"

In [25]:
df = pd.read_csv(TAXONOMY_CSV, dtype=str, low_memory=False)
print(f"Total de filas en gbif_taxonomy.csv: {len(df):,}")
print(f"Columnas: {list(df.columns)}")
print()

Total de filas en gbif_taxonomy.csv: 4,124,085
Columnas: ['species_key', 'scientific_name', 'canonical_name', 'rank', 'kingdom', 'phylum', 'class', 'order', 'family', 'genus', 'species', 'taxonomic_status', 'kingdom_key', 'phylum_key', 'class_key', 'order_key', 'family_key', 'genus_key']



In [26]:
# General

In [27]:
species_source = df[
    (df["rank"] == "species") &
    (df["taxonomic_status"] == "accepted")
].dropna(subset=["species_key", "canonical_name"])
species_source = species_source[
    (species_source["species_key"] != "") &
    (species_source["canonical_name"] != "")
]

In [28]:
total_source        = len(species_source)
unique_keys_source  = species_source["species_key"].nunique()

print(f"Filas rank=species y status=accepted en fuente: {total_source:,}")
print(f"species_key unicos en fuente:                   {unique_keys_source:,}")
print()

print("Especies por reino (fuente):")
print(species_source["kingdom"].value_counts().to_string())
print()

Filas rank=species y status=accepted en fuente: 2,552,973
species_key unicos en fuente:                   2,552,973

Especies por reino (fuente):
kingdom
Animalia          1810100
Plantae            441397
Fungi              171665
Chromista          100572
Bacteria            22976
Protozoa             4612
Archaea               870
incertae sedis        769
Viruses                12



In [29]:
# species_nodes.csv

In [30]:
if not SPECIES_CSV.exists():
    print(f"species_nodes.csv no encontrado en {SPECIES_CSV}")
    print("Corre extract_taxonomy_nodes.py primero.")
else:
    species_out = pd.read_csv(SPECIES_CSV, dtype=str, low_memory=False)

    total_out       = len(species_out)
    unique_keys_out = species_out["key"].nunique()

    print(f"Filas en species_nodes.csv:          {total_out:,}")
    print(f"keys unicos en species_nodes.csv:    {unique_keys_out:,}")
    print()

    print("Especies por reino (species_nodes.csv):")
    print(species_out["kingdom"].value_counts().to_string())
    print()

Filas en species_nodes.csv:          2,552,973
keys unicos en species_nodes.csv:    2,552,973

Especies por reino (species_nodes.csv):
kingdom
Animalia          1810100
Plantae            441397
Fungi              171665
Chromista          100572
Bacteria            22976
Protozoa             4612
Archaea               870
incertae sedis        769
Viruses                12



In [31]:
# validation

In [32]:
match_total = total_source == total_out
match_keys  = unique_keys_source == unique_keys_out

print(f"  Filas fuente:     {total_source:>10,}")
print(f"  Filas generado:   {total_out:>10,}")
print(f"  Coinciden:        {'SI' if match_total else 'NO -- diferencia de ' + str(abs(total_source - total_out))}")
print()
print(f"  Keys unicos fuente:    {unique_keys_source:>10,}")
print(f"  Keys unicos generado:  {unique_keys_out:>10,}")
print(f"  Coinciden:             {'SI' if match_keys else 'NO -- diferencia de ' + str(abs(unique_keys_source - unique_keys_out))}")
print()

  Filas fuente:      2,552,973
  Filas generado:    2,552,973
  Coinciden:        SI

  Keys unicos fuente:     2,552,973
  Keys unicos generado:   2,552,973
  Coinciden:             SI



In [33]:
# missing keys

In [35]:
if not match_keys:
    keys_source = set(species_source["species_key"].astype(str))
    keys_out    = set(species_out["key"].astype(str))

    only_in_source = keys_source - keys_out
    only_in_output = keys_out - keys_source

    if only_in_source:
        print(f"  Keys en fuente pero NO en generado ({len(only_in_source):,}):")
        for k in sorted(only_in_source)[:10]:
            print(f"    {k}")
        if len(only_in_source) > 10:
            print(f"    ... y {len(only_in_source) - 10:,} mas")

    if only_in_output:
        print(f"  Keys en generado pero NO en fuente ({len(only_in_output):,}):")
        for k in sorted(only_in_output)[:10]:
            print(f"    {k}")
        if len(only_in_output) > 10:
            print(f"    ... y {len(only_in_output) - 10:,} mas")